# Experiment 3 — does PV = NkT hold?

500 atoms in 3D at T* = 2.0, with the density set by `variable rho` in _pressure-3d.in_. The compressibility factor

$$Z = \frac{P}{\rho k_B T}$$

is exactly 1 for an ideal gas. Run ρ* = 0.01 and ρ* = 0.5 (and 0.1 if you have time) and **predict Z before each run**.

In [ ]:
# lammps-logfile is served from Atomify's own package index (no network needed).
%pip install -q lammps-logfile pandas

In [ ]:
import glob, os, time
import numpy as np
import lammps_logfile
import matplotlib.pyplot as plt

# Atomify stores every run of this project in runs/<run-name>/ next to this
# notebook (input snapshot, log.lammps, dumps). Pick the newest run here;
# use logs[0], logs[1], ... to look at older ones.
logs = sorted(glob.glob("runs/*/log.lammps"))
if not logs:
    raise RuntimeError("No runs yet: press Run in Atomify, wait for it to finish, then re-run this cell.")
print("Runs found:", *logs, sep="\n  ")

for attempt in range(5):
    try:
        log = lammps_logfile.File(logs[-1])
        break
    except FileNotFoundError:
        # Atomify may still be copying the finished run into the project.
        time.sleep(1)
        os.listdir(os.path.dirname(logs[-1]))
else:
    raise RuntimeError(f"{logs[-1]} is not readable yet: wait for the run to finish, then re-run this cell.")
print("Log keywords:", log.get_keywords())

Z against time for the newest run; the mean after the first 10 time units is the measurement:

In [ ]:
t = log.get("Time")
Z = log.get("v_Z")
P = log.get("Press")
eq = t > 10                                   # skip the equilibration

plt.figure(figsize=(8, 4))
plt.plot(t, Z, lw=1)
plt.axhline(1, color="gray", ls="--", label="ideal gas")
plt.xlabel("time t*"); plt.ylabel("Z = P / (ρ k T)"); plt.legend(); plt.show()
print(f"rho* = {log.get('v_rhoN')[0]:.3f}:   <Z> = {Z[eq].mean():.3f} ± {Z[eq].std() / np.sqrt(eq.sum()):.3f}     <P*> = {P[eq].mean():.4f}")

All runs of this project — Z against density:

In [ ]:
rhos, Zs = [], []
for path in logs:
    run = lammps_logfile.File(path)
    tt, ZZ = run.get("Time"), run.get("v_Z")
    rhos.append(run.get("v_rhoN")[0]); Zs.append(ZZ[tt > 10].mean())
plt.plot(rhos, Zs, "o-"); plt.axhline(1, color="gray", ls="--")
plt.xscale("log"); plt.xlabel("ρ*"); plt.ylabel("<Z>"); plt.show()

Note down:
* At which density is the gas ideal, and what does Z do as the density grows? Below 1 or above 1 — and what does each sign say about the forces between the atoms?
* PV = NkT came from atoms bouncing off the walls. What is missing from that argument when the atoms are close together?